In [4]:
import sys,os
sys.path.append('../')
os.environ["JAX_PLATFORMS"] = "cpu"
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np 
from geometry import (L2_distance,
                      build_get_similarities,
                      normalized_L2_distance,
                      build_information_imbalance,
                      mapped_compute_ranks,
                      )

def correlated_gaussian_batch(key, Ns, N, alpha):
    """
    Generate Ns pairs (x, y) of Gaussian vectors of length N.
    For each vector, entries of y with index/N < alpha are replaced with x's entries.
    """
    key_x, key_y = jax.random.split(key)
    
    # Independent Gaussian samples
    x = jax.random.normal(key_x, (Ns, N))
    y = jax.random.normal(key_y, (Ns, N))
    
    # Create mask: True where index/N < alpha
    rel_depth = jnp.linspace(0, 1, N, endpoint=False)
    mask = (rel_depth < alpha).astype(bool)
    
    # Broadcast mask to all samples
    mask = jnp.broadcast_to(mask, (Ns, N))
    
    # Replace y values where mask is True with x values
    y = jnp.where(mask, x, y)
    
    return x, y

def compute_II_for_alpha(
                        alpha, 
                        data_key, 
                        Ns, 
                        N, 
                        get_similarities, 
                        II_fn,
):
    """
    Generate data for a given alpha, compute correlations and information imbalance.
    Returns inf_imb and the standard deviation of the conditional rank distribution.
    """
    x, y = correlated_gaussian_batch(data_key, Ns, N, alpha)
    sim_X, sim_Y = get_similarities(x, y)

    R_II = mapped_compute_ranks(method="min")(sim_X, sim_Y)
    inf_imb, inf_imb_std = II_fn(R_II[0], R_II[1])

    return inf_imb, inf_imb_std


In [ ]:
# Example
master_seed = 12345
master_key = jax.random.PRNGKey(master_seed)
keyX, keyY = jax.random.split(master_key)
data_key = jax.random.PRNGKey(0)
key_distances = jax.random.PRNGKey(42)
key_distances, subkey_distances = jax.random.split(key_distances)
II_fn = build_information_imbalance(k=1) # JAX-Compiling the II function, keep k = 1.

In [15]:
Ns = 1000 # number of samples
get_similarities = build_get_similarities(key=subkey_distances, 
                                        sample_size=Ns, 
                                        similarity_fn=normalized_L2_distance,
                                        ) # compilation for the distance function

alpha = 0.3 # percentage of shared features
N = 100 # number of features
II, _ = compute_II_for_alpha(alpha, 
                    data_key, 
                    Ns, 
                    N, 
                    get_similarities, 
                    II_fn,
                    )
print(f'{II=}')
### Note that II.shape = (2,), because it contains  II(x-->y) and II(y-->x)

II=Array([0.52954217, 0.56951807], dtype=float64)
